In [11]:
import pandas as pd
import numpy as np
import pickle #Salvar tradutores

# TensorFlow e Keras (tive que seperar para evitar erro)
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences # type: ignore
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, LSTM, Dense # type: ignore
from tensorflow.keras.models import load_model # type: ignore

# Biblioteca do sklearn
from sklearn.preprocessing import LabelEncoder


# Parametros (tive que criar para mexer manualmente, sem ler o codigo completo)
vocab = 5000
maxlen = 20
path_dados = '../data/iniciacao.csv'
path_models = '../models/'

# Vetorização 
df = pd.read_csv(path_dados)

#pegar os valores direto
perguntas = df['perguntas'].values
respostas = df['respostas'].values

#print de teste
print(f"Encontradas {len(perguntas)} perguntas e {len(respostas)} respostas.")

#todo o processo de desenvolvimento da RNN será uma LSTM com o Keras API
tokenizer = Tokenizer(num_words=vocab, oov_token="<OOV>") #limite de palavras e o oov_token para palavras fora do vocabulario
tokenizer.fit_on_texts(perguntas)
x_perguntas = tokenizer.texts_to_sequences(perguntas) #transforma todas as palavras em sequencias numericas (coluna x)
x_processadas = pad_sequences(x_perguntas, maxlen=20, padding='post')

print("Exemplo de X (bruto):", perguntas[0])
print("Exemplo de X (processado):", x_processadas[0])

#Processo de Codificação Categórica
label_encoder = LabelEncoder()
y_respostas = label_encoder.fit_transform(np.array(respostas)) #transforma as respostas em numeros, aplicado o np.array para evitar erro
y_respostas = np.array(y_respostas)  # Força conversão para NumPy array, apenas para o vscode não reclamar
#aparentemente o LabelEncoder não aceita Series do pandas diretamente

print("Exemplo de Y (bruto):", respostas[0])
print("Exemplo de Y (Label):", y_respostas[0])

#vou converter os IDs para One-Hot Encoding, trabalhar apenas com o metodo binario, isso facilita a vida da rede neural, treinamento e categorização
num_classes = len(np.unique(y_respostas)) #calcular o output unico
y_categorico = to_categorical(y_respostas, num_classes=num_classes)

print(f"Total de classes (respostas únicas): {num_classes}")
print("Exemplo de Y (Categorical/One-Hot):", y_categorico[0])

with open(path_models + 'tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

with open(path_models + 'label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)


model = Sequential()

#Embedding transforma os IDs em vetores densos e é aqui que ela ira aprender o "significado" e contexto das palavras
model.add(Embedding(input_dim=vocab, 
                    output_dim=64))

#LSTM será a memoria para processar a sequência de vetores das palavras e de certa forma lembrar o contexto
model.add(LSTM(64))

#É a saida de decisão, pega os valores de LSTM e decide a resposta a retornar
model.add(Dense(units=num_classes, activation='softmax'))
model.summary() #mostrar arquitetura

#Treinar o Modelo

model.compile(loss='categorical_crossentropy', # Função de perda para classificação
              optimizer='adam',                 # Otimizador padrão
              metrics=['accuracy'])             # Queremos ver a acurácia

history = model.fit(x_processadas, 
                    y_categorico, 
                    epochs=50, 
                    batch_size=32, 
                    validation_split=0.1)

#Salvar o modelo que treinei
model.save('../models/chatbotIA.h5') #a forma usada é legada, mas funciona e provavelmente vou ter que mudar para o .keras futuramente


Encontradas 4 perguntas e 4 respostas.
Exemplo de X (bruto): oi ola hello
Exemplo de X (processado): [2 3 4 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
Exemplo de Y (bruto): Oi! Tudo bem?
Exemplo de Y (Label): 2
Total de classes (respostas únicas): 4
Exemplo de Y (Categorical/One-Hot): [0. 0. 1. 0.]


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 1.3879 - val_accuracy: 0.0000e+00 - val_loss: 1.4289
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.3333 - loss: 1.3727 - val_accuracy: 0.0000e+00 - val_loss: 1.4770
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.3333 - loss: 1.3579 - val_accuracy: 0.0000e+00 - val_loss: 1.5277
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.3333 - loss: 1.3434 - val_accuracy: 0.0000e+00 - val_loss: 1.5829
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.3333 - loss: 1.3286 - val_accuracy: 0.0000e+00 - val_loss: 1.6448
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.3333 - loss: 1.3132 - val_accuracy: 0.0000e+00 - val_loss: 1.7164
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 0.3333 - loss: 1.2969 - val_accuracy: 0.0000e+00 - val_loss: 1.8009
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.3333 - loss: 1.2795 - val_accuracy: 